In [ ]:
!pip -q install jupyter_bokeh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.5 MB/s eta 0:00:00


### Must watch from TEST and FINAL's perspective

  - Momentum : https://www.youtube.com/watch?v=Vce8w1sy0e8
  - RMSProp  : https://www.youtube.com/watch?v=ajI_HTyaCu8&t=102s
  - Adam     : https://www.youtube.com/watch?v=tuU59-G1PgU




# Defining Loss Terrain

In [ ]:
import torch
import torch.nn.functional as F

class LossFunction:
    def __init__(self, A_g, sigma_g, global_min_pos, local_min_params, noise_factor=0.1):
        self.A_g = A_g
        self.sigma_g = sigma_g
        self.global_min_pos = global_min_pos
        self.local_min_params = local_min_params
        self.noise_factor = noise_factor

    def generate_terrain_mesh(self, x_range, y_range):
        """Generates a terrain mesh."""
        # Create a meshgrid of x and y values using PyTorch
        x = torch.linspace(x_range[0], x_range[1], 100)
        y = torch.linspace(y_range[0], y_range[1], 100)
        X, Y = torch.meshgrid(x, y, indexing='ij')

        # Return the loss calculated at each (X, Y) point
        return X, Y, self.get_loss(X, Y)

    def get_loss(self, X, Y):
        """Calculates the loss."""
        # Global minimum: Gaussian function
        Z = self.A_g * torch.exp(-((X - self.global_min_pos[0])**2 + (Y - self.global_min_pos[1])**2) / (2 * self.sigma_g**2))

        # Local minima: Sum of Gaussian functions
        for A_l, sigma_l, x_l, y_l in self.local_min_params:
            Z += A_l * torch.exp(-((X - x_l)**2 + (Y - y_l)**2) / (2 * sigma_l**2))

        # Add random noise
        Z += self.noise_factor * torch.randn_like(X)

        return Z

# Visualizing the terrain

In [ ]:
import numpy as np
import plotly.graph_objs as go
import param
import panel as pn
import torch

# Enable Panel extension for Plotly
pn.extension('plotly')

class TerrainModel(param.Parameterized):
    # Define parameters for the model
    x_min = param.Number(-10, bounds=(-20, 0), doc="Minimum x-value")
    x_max = param.Number(10, bounds=(0, 20), doc="Maximum x-value")
    y_min = param.Number(-10, bounds=(-20, 0), doc="Minimum y-value")
    y_max = param.Number(10, bounds=(0, 20), doc="Maximum y-value")

    A_g = param.Number(-10, bounds=(-20, 0), doc="Amplitude of global minimum")
    sigma_g = param.Number(2, bounds=(0.1, 5), doc="Aperture size of global minimum")
    global_min_x = param.Number(0, bounds=(-10, 10), doc="x-position of global minimum")
    global_min_y = param.Number(0, bounds=(-10, 10), doc="y-position of global minimum")

    # Input box for local minima as list of tuples
    local_min_params = param.String("(-5, 1, 3, 3), (-7, 1.5, -4, -4), (-4, 0.8, -2, 5)", doc="Local minima parameters as list of tuples")

    # Set the step size for noise_factor to 0.01
    noise_factor = param.Number(0.00, bounds=(0, 1), step=0.01, doc="Noise factor")

    # Update button
    update_plot = param.Action(lambda x: x.param.trigger('update_plot'), label="Update Plot")

    def parse_local_min_params(self):
        """
        Parse the input string for local minima parameters and return as a list of tuples.
        """
        try:
            # Safely evaluate the input string to convert it into a list of tuples
            local_min_params = eval(self.local_min_params)
            # Ensure it's a list of tuples
            if isinstance(local_min_params, tuple):
                local_min_params = [*local_min_params]
            return local_min_params
        except:
            return []

    @param.depends('update_plot')
    def generate_plot(self):
        # Define the ranges for x and y
        x_range = (self.x_min, self.x_max)
        y_range = (self.y_min, self.y_max)

        # Define global minimum position
        global_min_pos = (self.global_min_x, self.global_min_y)

        # Parse the local minima parameters from the string input
        local_min_params = self.parse_local_min_params()

        if not local_min_params:
            return go.Figure().update_layout(title="Invalid local minima parameters")

        # Generate mesh using the updated parameters
        terrain_loss = LossFunction(self.A_g, self.sigma_g, global_min_pos, local_min_params, self.noise_factor)
        X, Y, Z = terrain_loss.generate_terrain_mesh(x_range, y_range)
        X, Y, Z = X.numpy(), Y.numpy(), Z.numpy()

        # Create the surface plot (3D)
        surface = go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')

        # Create the figure and add the surface (3D plot)
        fig_3d = go.Figure(data=[surface])
        fig_3d.update_layout(scene=dict(
                            xaxis_title='X Axis',
                            yaxis_title='Y Axis',
                            zaxis_title='Z Axis'),
                          title="3D Terrain-like Mesh Plot")

        # Create the contour plot (2D)
        contour = go.Contour(z=Z.flatten(), x=X.flatten(), y=Y.flatten(), colorscale='Viridis', contours=dict(showlabels=True))

        # Create the figure for the contour plot
        fig_contour = go.Figure(data=[contour])
        fig_contour.update_layout(
            title="Contour Plot",
            xaxis_title="X Axis",
            yaxis_title="Y Axis"
        )

        # Return both plots in a Panel layout
        return pn.Row(fig_3d, fig_contour)

# Instantiate the TerrainModel
terrain_model = TerrainModel()

# Create Panel dashboard with dynamic controls and live plot update, now with 3 columns
pn.Row(terrain_model.param, terrain_model.generate_plot).servable()

Row
    [0] Column(margin=(5, 10), name='TerrainModel')
        [0] StaticText(value='<b>TerrainModel</b>')
        [1] FloatSlider(end=0, label='X min', name='X min', start=-20, value=-10)
        [2] FloatSlider(end=20, label='X max', name='X max', value=10)
        [3] FloatSlider(end=0, label='Y min', name='Y min', start=-20, value=-10)
        [4] FloatSlider(end=20, label='Y max', name='Y max', value=10)
        [5] FloatSlider(end=0, label='A g', name='A g', start=-20, value=-10)
        [6] FloatSlider(end=5, label='Sigma g', name='Sigma g', start=0.1, value=2)
        [7] FloatSlider(end=10, label='Global min x', name='Global min x', start=-10)
        [8] FloatSlider(end=10, label='Global min y', name='Global min y', start=-10)
        [9] TextInput(description='Local minima p..., label='Local min params', name='Local min params', value='(-5, 1, 3, 3), ...)
        [10] FloatSlider(label='Noise factor', name='Noise factor', step=0.01)
        [11] Button(label='Update Plot', name='Update Plot')
    [1] ParamMethod(method, _pane=Row, defer_load=False)

# Defining dummy pytorch optization pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

class TerrainOptimizer(nn.Module):
    def __init__(self, terrain_loss, x0, x1, optimizer=optim.SGD, **optimargs):
        super(TerrainOptimizer, self).__init__()

        # Initialize the two parameters x0, x1 randomly within the x_range and y_range
        self.x0 = nn.Parameter(torch.tensor([x0], dtype=torch.float))
        self.x1 = nn.Parameter(torch.tensor([x1], dtype=torch.float))

        # Store the loss function
        self.terrain_loss = terrain_loss

        # Set up the optimizer
        self.optimizer = optimizer(self.parameters(), **optimargs)

    def forward(self):
        # Forward pass to compute the loss
        X = self.x0.view(-1, 1)  # Reshape to match meshgrid format
        Y = self.x1.view(-1, 1)
        Z = self.terrain_loss.get_loss(X, Y)
        return Z.mean()  # Return the mean loss for optimization

    def optimize(self, log_iter, n_iter=100, verbose = False):
        trace = []  # Store the trace of x0, x1, and loss

        for i in tqdm(range(n_iter)):
            self.optimizer.zero_grad()  # Zero gradients
            loss = self.forward()  # Forward pass
            loss.backward()  # Backward pass
            self.optimizer.step()  # Update parameters
            if i%log_iter == 0:
              # Log the intermediate values
              trace.append((self.x0.item(), self.x1.item(), loss.item()))
              if verbose:
                print(f"\nIteration {i+1}, x0: {self.x0.item():.4f}, x1: {self.x1.item():.4f}, loss: {loss.item():.4f}")

        return trace

# Example usage:
terrain_loss = LossFunction(
    A_g=-10,
    sigma_g=2,
    global_min_pos=(0, 0),
    local_min_params=[(-5, 1, 3, 3), (-7, 1.5, -4, -4), (-4, 0.8, -2, 5)],
    noise_factor=0.0
)

# Instantiate the optimizer module
model = TerrainOptimizer(
    terrain_loss,
    x0=1,  # Initial value for x0
    x1=1,  # Initial value for x1
    optimizer=optim.RMSprop,  # Use Adam optimizer
    lr=0.001  # Learning rate for the optimizer
)

# Perform optimization
trace = model.optimize(log_iter = 1000, n_iter=10000, verbose = True)

  2%|▏         | 185/10000 [00:00<00:10, 918.68it/s]


Iteration 1, x0: 0.9900, x1: 0.9900, loss: -7.8797


 11%|█         | 1109/10000 [00:01<00:10, 883.67it/s]


Iteration 1001, x0: 0.0612, x1: 0.0612, loss: -9.9960


 21%|██▏       | 2139/10000 [00:02<00:08, 914.08it/s]


Iteration 2001, x0: -0.0034, x1: -0.0034, loss: -10.0064


... (11 outputs left out)


Iteration 8001, x0: -0.0034, x1: -0.0034, loss: -10.0064


 92%|█████████▏| 9150/10000 [00:11<00:01, 814.29it/s]


Iteration 9001, x0: -0.0034, x1: -0.0034, loss: -10.0064


100%|██████████| 10000/10000 [00:12<00:00, 807.88it/s]


In [ ]:
trace

[(0.9900000095367432, 0.9900000095367432, -7.879690170288086),
 (0.061170242726802826, 0.06117025762796402, -9.99597454071045),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164),
 (-0.0033810450695455074, -0.0033810425084084272, -10.006357192993164)]

# Visualizing the optimization path

In [ ]:
import numpy as np
import plotly.graph_objs as go
import param
import panel as pn
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

# Enable Panel extension for Plotly
pn.extension('plotly')

class TerrainModel(param.Parameterized):
    x_min = param.Number(-10, bounds=(-20, 0), doc="Minimum x-value")
    x_max = param.Number(10, bounds=(0, 20), doc="Maximum x-value")
    y_min = param.Number(-10, bounds=(-20, 0), doc="Minimum y-value")
    y_max = param.Number(10, bounds=(0, 20), doc="Maximum y-value")

    A_g = param.Number(-10, bounds=(-20, 0), doc="Amplitude of global minimum")
    sigma_g = param.Number(2, bounds=(0.1, 5), doc="Aperture size of global minimum")
    global_min_x = param.Number(0, bounds=(-10, 10), doc="x-position of global minimum")
    global_min_y = param.Number(0, bounds=(-10, 10), doc="y-position of global minimum")
    local_min_params = param.String("(-5, 1, 3, 3), (-7, 1.5, -4, -4), (-4, 0.8, -2, 5)", doc="Local minima parameters as list of tuples")

    noise_factor = param.Number(0.00, bounds=(0, 1), step=0.01, doc="Noise factor")

    optimizer = param.ObjectSelector(default='SGD', objects=['SGD', 'Adam', 'RMSprop', 'Adagrad', 'AdamW', 'Adadelta',])
    optim_args = param.String('{"lr": 0.01}', doc="Optimizer arguments in Python dict format")
    start_point = param.Tuple((1.9, -2.9), doc="Starting point (x0, y0)")
    n_iter = param.Integer(10000, bounds=(100, 100000), doc="Number of iterations")
    log_iter = param.Integer(1000, bounds=(100, 10000), doc="Logging interval")

    update_plot = param.Action(lambda x: x.param.trigger('update_plot'), label="Update Plot")

    def parse_local_min_params(self):
        try:
            local_min_params = eval(self.local_min_params)
            if isinstance(local_min_params, tuple):
                local_min_params = [*local_min_params]
            return local_min_params
        except:
            return []

    def parse_optim_args(self):
        try:
            return eval(self.optim_args)
        except:
            return {}

    @param.depends('update_plot')
    def generate_plot(self):
        x_range = (self.x_min, self.x_max)
        y_range = (self.y_min, self.y_max)
        global_min_pos = (self.global_min_x, self.global_min_y)
        local_min_params = self.parse_local_min_params()
        optim_args = self.parse_optim_args()

        if not local_min_params:
            return go.Figure().update_layout(title="Invalid local minima parameters")

        terrain_loss = LossFunction(self.A_g, self.sigma_g, global_min_pos, local_min_params, self.noise_factor)
        X, Y, Z = terrain_loss.generate_terrain_mesh(x_range, y_range)
        X, Y, Z = X.numpy(), Y.numpy(), Z.numpy()

        # Create 3D mesh plot
        fig_3d = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')])
        fig_3d.update_layout(scene=dict(xaxis_title='X Axis', yaxis_title='Y Axis', zaxis_title='Z Axis'),
                             title="3D Terrain-like Mesh Plot",
                             height=800)  # Adjust the height for 3D plot

        # Create contour plot
        contour = go.Contour(z=Z.flatten(), x=X.flatten(), y=Y.flatten(), colorscale='Viridis', contours=dict(showlabels=True))
        fig_contour = go.Figure(data=[contour])
        fig_contour.update_layout(title="Contour Plot", xaxis_title="X Axis", yaxis_title="Y Axis", height=800)  # Adjust the height for contour plot

        # Optimizer part
        start_x, start_y = self.start_point
        optimizer_class = getattr(optim, self.optimizer)
        terrain_optimizer = TerrainOptimizer(terrain_loss, start_x, start_y, optimizer=optimizer_class, **optim_args)
        trace = terrain_optimizer.optimize(self.log_iter, self.n_iter)

        trace_x, trace_y, trace_z = zip(*[(x, y, terrain_loss.get_loss(torch.tensor([[x]]), torch.tensor([[y]])).item()) for x, y, _ in trace])

        # Add trace to 3D plot with lines to represent movement
        fig_3d.add_trace(go.Scatter3d(x=trace_x, y=trace_y, z=trace_z, mode='markers+lines', marker=dict(size=3, color='red'), name="Optimization Trace"))

        # Add trace to Contour plot with lines for direction
        fig_contour.add_trace(go.Scatter(x=trace_x, y=trace_y, mode='lines+markers', line=dict(color='red'), marker=dict(size=6, color='red'), name="Optimization Trace"))

        return pn.Row(fig_3d, fig_contour)

# Instantiate the TerrainModel
terrain_model = TerrainModel()

# Create Panel dashboard with dynamic controls and live plot update, now with optimizer controls
pn.Row(terrain_model.param, terrain_model.generate_plot).servable()

100%|██████████| 10000/10000 [00:11<00:00, 903.74it/s]


Row
    [0] Column(margin=(5, 10), name='TerrainModel')
        [0] StaticText(value='<b>TerrainModel</b>')
        [1] FloatSlider(end=0, label='X min', name='X min', start=-20, value=-10)
        [2] FloatSlider(end=20, label='X max', name='X max', value=10)
        [3] FloatSlider(end=0, label='Y min', name='Y min', start=-20, value=-10)
        [4] FloatSlider(end=20, label='Y max', name='Y max', value=10)
        [5] FloatSlider(end=0, label='A g', name='A g', start=-20, value=-10)
        [6] FloatSlider(end=5, label='Sigma g', name='Sigma g', start=0.1, value=2)
        [7] FloatSlider(end=10, label='Global min x', name='Global min x', start=-10)
        [8] FloatSlider(end=10, label='Global min y', name='Global min y', start=-10)
        [9] TextInput(description='Local minima p..., label='Local min params', name='Local min params', value='(-5, 1, 3, 3), ...)
        [10] FloatSlider(label='Noise factor', name='Noise factor', step=0.01)
        [11] Select(label='Optimizer', name='Optimizer', options=OrderedDict({'SGD': 'SGD',...]), value='SGD')
        [12] TextInput(description='Optimizer arguments i..., label='Optim args', name='Optim args', value='{"lr": 0.01}')
        [13] TupleInput(description='Starting point (x0, y0)', label='Start point', name='Start point', type=<class 'tuple'>, value=(1.9, -2.9))
        [14] IntSlider(end=100000, label='N iter', name='N iter', start=100, value=10000)
        [15] IntSlider(end=10000, label='Log iter', name='Log iter', start=100, value=1000)
        [16] Button(label='Update Plot', name='Update Plot')
    [1] ParamMethod(method, _pane=Row, defer_load=False)

100%|██████████| 10000/10000 [00:13<00:00, 716.12it/s]
/usr/local/lib/python3.13/dist-packages/panel/viewable.py:529: BokehUserWarning:

reference already known '2c4199de-9e35-4a36-ac8b-94475f299fa7'

100%|██████████| 10000/10000 [00:16<00:00, 602.64it/s]
/usr/local/lib/python3.13/dist-packages/panel/viewable.py:529: BokehUserWarning:

reference already known '8d5ca8ad-4a01-4b9c-ac13-822ff37ab24a'

100%|██████████| 10000/10000 [00:14<00:00, 703.26it/s]
/usr/local/lib/python3.13/dist-packages/panel/viewable.py:529: BokehUserWarning:

reference already known '5ed43217-f6cc-4df2-bad3-5604396cbfac'

